# Reward Models & Reward Hacking

RLHF works by replacing an objective nobody can write down — "be helpful" — with one a
neural network learned from comparisons. That substitution is the whole trick, and it is
also the whole problem: **you then optimise against the approximation, not the thing you
wanted.**

Push hard enough on any learned proxy and the policy finds the region where the proxy is
wrong. This is not a bug in RLHF; it is Goodhart's law with a gradient. The measurable
consequence is an **inverted-U**: as you optimise the reward model harder, true quality
rises, peaks, and then falls while the reward model's score keeps climbing.

This notebook trains a reward model from preferences, then reproduces that curve and the
standard control for it. Follows [PPO](ppo-from-scratch.ipynb); contrast with the
verifiable rewards in [GRPO & RLVR](grpo-rlvr.ipynb).

## 1. What & Why

A reward model turns pairwise preferences into a scalar. The standard formulation is
**Bradley–Terry**: assume

```
P(A preferred to B) = σ(r(A) − r(B))
```

and fit `r` by maximum likelihood on human comparisons. Note what this gives you: a
reward that is only defined **up to an additive constant**, and only calibrated *in the
region the comparisons came from*.

**Why comparisons rather than direct ratings.** People are poor at absolute scoring and
much better at relative judgement — the same reason
[pairwise LLM judging](../12-model-evaluation/llm-as-judge.ipynb) beats pointwise scoring.
Ratings drift between annotators and over time; comparisons do not.

**Where it breaks.** The reward model is trained on outputs from some policy. When RL
moves the policy, the outputs drift out of that distribution and the reward model's
predictions become extrapolation. The policy is explicitly *searching* for high reward, so
it finds exactly the places where the extrapolation is most wrong.

**The standard control** is a KL penalty to the reference policy: stay near the
distribution where the reward model is valid. That is a genuine fix, and it is also a
tax — you are trading the improvement you could get against the risk of leaving the
region where your proxy means anything.

## 2. Mental Model

**A map drawn from one traveller's journey.**

The reward model is a map of quality, drawn from the routes annotators actually walked. On
those routes it is accurate. A step off them and it is guesswork, drawn from whatever
pattern the model inferred — and the further you go, the more confidently wrong it gets.

RL is a search for the highest point *on the map*. It does not know the map's edges. If
the map has a spurious peak in unexplored terrain — a place where the model learned
"longer answers score higher" and never saw a counterexample — the policy will find it,
climb it, and report enormous success.

Two consequences follow, and both are practical:

- **Reward going up is not evidence of anything** once you are far from the training
  distribution. The number is the map's opinion, and you are off the map.
- **KL to the reference policy is your distance from surveyed ground.** That is why it is
  the right thing to penalise: not because moving is bad, but because the map's
  reliability decays with distance.

The failure has a signature: reward model score rising smoothly while human evaluation
falls. If you only measure the first, everything looks excellent.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Bradley–Terry** | `P(A ≻ B) = σ(r(A) − r(B))`. The standard preference likelihood. |
| **Shift invariance** | Only reward *differences* are identified; `r` and `r + c` fit equally well. Normalise before use. |
| **Proxy vs true reward** | What the model scores versus what you actually wanted. Optimisation widens the gap. |
| **Goodhart's law** | "When a measure becomes a target, it ceases to be a good measure." The whole subject in one sentence. |
| **Over-optimisation** | Continuing to improve the proxy past the point where true quality declines. |
| **KL penalty / coefficient `β`** | `r_total = r_RM − β·KL(π ‖ π_ref)`. The distance-from-surveyed-ground tax. |
| **Reward hacking** | Exploiting a systematic error in the reward model — length, formatting, sycophancy, confident tone. |
| **Length bias** | The best-documented instance: reward models reliably prefer longer responses beyond what quality justifies. |
| **Ensemble / uncertainty** | Several reward models; penalise disagreement, which is high exactly where extrapolation is happening. |
| **Iterated RLHF** | Re-collect preferences on the *new* policy's outputs and retrain the RM. Re-surveying the map. |
| **Preference-free alternatives** | [Verifiable rewards](grpo-rlvr.ipynb), where the reward is checked rather than predicted. |

## 4. Setup

NumPy. Reward models are fitted here as logistic regressions on features, which is the
Bradley–Terry model exactly — a deep RM is the same objective with a learned feature map,
and every pathology below is a property of the objective rather than the architecture.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — fit a reward model from preferences

Bradley–Terry by gradient ascent on the log-likelihood. With enough comparisons it
recovers the true quality function up to a constant.

In [2]:
D_FEAT = 6
# The TRUE quality function. Feature 5 is 'length' and genuinely contributes nothing.
w_true = np.array([1.0, 0.8, -0.6, 0.4, 0.3, 0.0])
CURV = 0.20      # real quality is CONCAVE: every dimension has a best level, not "more"

def true_reward(x):
    # Quality with an interior optimum -- more detail, more hedging and more
    # enthusiasm all stop helping past some point. This is what makes a linear
    # reward model an approximation rather than the truth.
    return x @ w_true - CURV * np.sum(x ** 2, axis=-1)

def sample_responses(n, seed):
    r = np.random.default_rng(seed)
    x = r.normal(0, 1, (n, D_FEAT))
    x[:, 5] = r.normal(0, 1, n)          # length, uncorrelated with quality
    return x

def make_preferences(x, n_pairs, w, noise, seed):
    '''Humans compare pairs; their judgements are noisy Bradley-Terry draws.'''
    r = np.random.default_rng(seed)
    i, j = r.integers(0, len(x), n_pairs), r.integers(0, len(x), n_pairs)
    diff = true_reward(x[i]) - true_reward(x[j])
    p = 1 / (1 + np.exp(-(diff / noise)))
    wins = r.random(n_pairs) < p
    return i, j, wins

def fit_reward_model(x, i, j, wins, steps=3000, lr=0.5, l2=1e-3):
    w = np.zeros(D_FEAT)
    d = x[i] - x[j]
    y = wins.astype(float)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(d @ w)))
        grad = d.T @ (y - p) / len(y) - l2 * w
        w += lr * grad
    return w

x_pool = sample_responses(4000, seed=1)
print(f"{'preference pairs':>17} {'recovered w (normalised)':>44} {'cosine to truth':>16}")
for n_pairs in (200, 1000, 5000, 40000):
    i, j, wins = make_preferences(x_pool, n_pairs, w_true, noise=1.0, seed=n_pairs)
    w_hat = fit_reward_model(x_pool, i, j, wins)
    w_n = w_hat / np.linalg.norm(w_hat)
    cos = float(w_n @ (w_true / np.linalg.norm(w_true)))
    print(f"{n_pairs:17d} {str(np.round(w_n, 3)):>44} {cos:16.3f}")

print(f"\ntrue direction:   {np.round(w_true / np.linalg.norm(w_true), 3)}")
print("\nWith enough comparisons the reward model recovers the true quality direction.")
print("Note it can only ever recover a DIRECTION -- Bradley-Terry is shift- and")
print("scale-invariant, since only differences enter the likelihood. That is why reward")
print("model scores are meaningless in absolute terms and are always normalised.")

 preference pairs                     recovered w (normalised)  cosine to truth
              200  [ 0.617  0.417 -0.551  0.369  0.075 -0.031]            0.967
             1000  [ 0.64   0.507 -0.457  0.274  0.215  0.051]            0.996


             5000  [ 0.675  0.525 -0.401  0.276  0.174  0.025]            0.999


            40000  [ 0.657  0.547 -0.401  0.269  0.19  -0.02 ]            1.000

true direction:   [ 0.667  0.533 -0.4    0.267  0.2    0.   ]

With enough comparisons the reward model recovers the true quality direction.
Note it can only ever recover a DIRECTION -- Bradley-Terry is shift- and
scale-invariant, since only differences enter the likelihood. That is why reward
model scores are meaningless in absolute terms and are always normalised.


### Example 2 — over-optimisation: the inverted U

Now the central phenomenon. Fit a reward model on **limited** data so it is imperfect, then
optimise against it with increasing strength and track *both* the proxy score and the true
quality.

In [3]:
# A reward model fitted on the reference distribution. It is a good LOCAL linear
# approximation to a truth that is not linear -- the realistic situation.
i, j, wins = make_preferences(x_pool, 3000, w_true, noise=1.0, seed=42)
w_proxy = fit_reward_model(x_pool, i, j, wins)

print(f"proxy w: {np.round(w_proxy, 3)}")
print("  a good local fit -- but the truth it approximates is concave, so the fit is")
print("  only valid near where the comparisons were collected.\n")

direction = w_proxy / np.linalg.norm(w_proxy)
x_ref = sample_responses(3000, seed=7)

print(f"{'KL-ish distance':>16} {'proxy reward':>14} {'TRUE reward':>13}")
best_true, best_at = -1e9, None
for strength in (0.0, 1.0, 2.0, 3.0, 3.75, 5.0, 6.0, 8.0, 10.0):
    x_new = x_ref + strength * direction
    proxy = float(np.mean(x_new @ w_proxy))
    true = float(np.mean(true_reward(x_new)))
    if true > best_true:
        best_true, best_at = true, strength
    print(f"{strength:16.2f} {proxy:14.3f} {true:13.3f}")

print(f"\ntrue quality peaks at distance {best_at} and then DECLINES -- ending far BELOW")
print("where it started -- while the proxy score rises monotonically throughout.")
print("\nThat divergence is over-optimisation. Past the peak, every additional unit of")
print("reward-model score is bought by moving further into the region where the reward")
print("model is wrong. If you monitor only the proxy, this looks like uninterrupted")
print("progress; the run that produced it is getting worse.")

proxy w: [ 0.852  0.736 -0.533  0.371  0.298  0.004]
  a good local fit -- but the truth it approximates is concave, so the fit is
  only valid near where the comparisons were collected.

 KL-ish distance   proxy reward   TRUE reward
            0.00         -0.019        -1.203
            1.00          1.314         0.101
            2.00          2.648         1.005
            3.00          3.982         1.510
            3.75          4.982         1.626
            5.00          6.649         1.319
            6.00          7.983         0.623
            8.00         10.650        -1.968
           10.00         13.317        -6.159

true quality peaks at distance 3.75 and then DECLINES -- ending far BELOW
where it started -- while the proxy score rises monotonically throughout.

That divergence is over-optimisation. Past the peak, every additional unit of
reward-model score is bought by moving further into the region where the reward
model is wrong. If you monitor only the prox

### Example 3 — the KL penalty, and what it costs

The standard control. Penalise distance from the reference policy, and the optimum stops
short of the cliff — at the price of some achievable gain.

In [4]:
def best_under_penalty(beta, grid=np.linspace(0, 10, 200)):
    '''Choose the distance maximising (proxy reward - beta * distance^2), then report
    the TRUE reward actually achieved there. distance^2 stands in for KL.'''
    best = None
    for s in grid:
        xn = x_ref + s * direction
        proxy = float(np.mean(xn @ w_proxy))
        objective = proxy - beta * s ** 2
        if best is None or objective > best[0]:
            best = (objective, s, proxy, float(np.mean(true_reward(xn))))
    return best

print(f"{'beta (KL coef)':>15} {'chosen distance':>16} {'proxy':>9} {'TRUE':>9}")
for beta in (0.0, 0.05, 0.1, 0.2, 0.5, 1.0):
    _, s, proxy, true = best_under_penalty(beta)
    print(f"{beta:15.2f} {s:16.2f} {proxy:9.3f} {true:9.3f}")
print(f"\n(the unconstrained true optimum was {best_true:.3f}, at distance {best_at})")

print("\nbeta = 0 runs to the edge and lands far past the peak, ending with TRUE reward")
print("well below where it started. A moderate beta stops close to the true optimum.")
print("Too large a beta barely moves at all and leaves real improvement unclaimed.")
print("\nSo beta is not a safety switch to be maximised -- it is a bias/variance dial on")
print("how much you trust your reward model. The right value depends on how good the")
print("reward model is, which is why it needs tuning per RM rather than per task, and")
print("why it must be re-tuned whenever the RM is retrained.")

 beta (KL coef)  chosen distance     proxy      TRUE
           0.00            10.00    13.317    -6.159
           0.05            10.00    13.317    -6.159
           0.10             6.68     8.894    -0.082
           0.20             3.32     4.404     1.586
           0.50             1.36     1.790     0.470
           1.00             0.65     0.852    -0.306

(the unconstrained true optimum was 1.626, at distance 3.75)

beta = 0 runs to the edge and lands far past the peak, ending with TRUE reward
well below where it started. A moderate beta stops close to the true optimum.
Too large a beta barely moves at all and leaves real improvement unclaimed.

So beta is not a safety switch to be maximised -- it is a bias/variance dial on
how much you trust your reward model. The right value depends on how good the
reward model is, which is why it needs tuning per RM rather than per task, and
why it must be re-tuned whenever the RM is retrained.


### Example 4 — reward hacking through a spurious feature

Over-optimisation in its most recognisable form. If the reward model picks up a feature
that correlates with quality in the training data but does not cause it, the policy will
exploit that feature and nothing else.

In [5]:
# Now make LENGTH correlate with quality in the annotation data -- as it does in reality,
# because more thorough answers really are often longer.
def sample_with_length_confound(n, seed):
    r = np.random.default_rng(seed)
    x = r.normal(0, 1, (n, D_FEAT))
    quality = x[:, :5] @ w_true[:5]
    x[:, 5] = 0.8 * quality + r.normal(0, 0.6, n)     # length tracks quality...
    return x

x_conf = sample_with_length_confound(6000, seed=3)
i, j, wins = make_preferences(x_conf, 6000, w_true, noise=1.0, seed=9)

# The crucial ingredient is PARTIAL OBSERVABILITY. A real reward model sees text, not
# ground-truth "clarity" and "relevance" scores -- so a visible feature that correlates
# with the hidden ones becomes its stand-in for them.
OBSERVED = [0, 1, 5]        # helpful, accurate, LENGTH -- but not verbose/clear/relevant
names = ["helpful", "accurate", "verbose-bad", "clear", "relevant", "LENGTH"]

d = (x_conf[i] - x_conf[j])[:, OBSERVED]
w_obs = np.zeros(len(OBSERVED))
for _ in range(3000):
    pr = 1 / (1 + np.exp(-(d @ w_obs)))
    w_obs += 0.5 * (d.T @ (wins.astype(float) - pr) / len(wins) - 1e-3 * w_obs)

print("the reward model can only see 3 of the 6 quality dimensions:\n")
print(f"{'feature':14} {'true weight':>12} {'RM weight':>12}")
for k, nm in enumerate(names):
    seen = k in OBSERVED
    cell = f"{w_obs[OBSERVED.index(k)]:12.2f}" if seen else "    (unseen)"
    print(f"{nm:14} {w_true[k]:12.2f} {cell}")

print("\nLENGTH truly contributes NOTHING (true weight 0.00), yet the reward model gives")
print("it the LARGEST weight of the three it can see -- because length correlates with")
print("the dimensions it cannot see, making it the best available stand-in for them.\n")

def rm_score(x):
    return x[:, OBSERVED] @ w_obs

def real_quality(x):
    return x[:, :5] @ w_true[:5]

base = sample_responses(3000, seed=21)
print(f"{'policy':34} {'RM score':>10} {'TRUE quality':>13}")
for label, delta in [("reference", None), ("improve real quality", "quality"),
                     ("increase LENGTH only", "length")]:
    xn = base.copy()
    if delta == "quality":
        xn[:, :5] += 1.5 * (w_true[:5] / np.linalg.norm(w_true[:5]))
    elif delta == "length":
        xn[:, 5] += 3.0
    print(f"{label:34} {float(np.mean(rm_score(xn))):10.3f} "
          f"{float(np.mean(real_quality(xn))):13.3f}")

print("\nPadding length alone scores HIGHER on the reward model than genuinely improving")
print("quality does, while true quality does not move at all. A policy optimising this")
print("reward model will find that immediately: it is by far the cheapest gain available.")
print("\nThis is the mechanism behind the best-documented RLHF pathology -- responses")
print("growing steadily longer over training with no improvement in helpfulness -- and")
print("it is why length-controlled evaluation exists.")

the reward model can only see 3 of the 6 quality dimensions:

feature         true weight    RM weight
helpful                1.00         0.38
accurate               0.80         0.31
verbose-bad           -0.60     (unseen)
clear                  0.40     (unseen)
relevant               0.30     (unseen)
LENGTH                 0.00         0.37

LENGTH truly contributes NOTHING (true weight 0.00), yet the reward model gives
it the LARGEST weight of the three it can see -- because length correlates with
the dimensions it cannot see, making it the best available stand-in for them.

policy                               RM score  TRUE quality
reference                              -0.005        -0.004
improve real quality                    0.627         2.246
increase LENGTH only                    1.113        -0.004

Padding length alone scores HIGHER on the reward model than genuinely improving
quality does, while true quality does not move at all. A policy optimising this
reward mod

## 6. Gotchas & Pitfalls

- **Treating the reward model's score as progress.** Example 2. Past the peak it is
  actively anti-correlated with what you want. Always hold out a human or verifiable
  evaluation.
- **No KL penalty, or one tuned once and never revisited.** `β` depends on how good the
  reward model is; retraining the RM changes the right value.
- **Comparing reward scores across reward models, or across training runs.** Bradley–Terry
  is shift- and scale-invariant (Example 1), so absolute values mean nothing.
- **Annotating only on-policy outputs from one model.** The RM is then valid on a very
  narrow distribution, and RL immediately leaves it.
- **Ignoring length.** Example 4. Length is the most reliable confound in preference data.
  Control for it — length-balanced training pairs, or an explicit length penalty.
- **A single reward model.** An ensemble gives you an uncertainty signal that is high
  exactly where extrapolation is happening, and penalising disagreement is one of the few
  controls that targets the actual failure.
- **Assuming more preference data fixes it.** It raises the peak and moves it right, but
  the qualitative shape persists — any finite dataset leaves regions unconstrained.
- **Sycophancy as a special case.** Agreeing with the user is rewarded by annotators
  slightly and by the resulting model strongly. It is length bias with a different
  feature.
- **Forgetting that annotators disagree.** Inter-annotator agreement on nuanced
  comparisons is often modest, which caps how good any reward model can be.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Open-ended helpfulness, no checkable answer | **Reward model + [PPO](ppo-from-scratch.ipynb)** with a KL penalty |
| Maths, code, anything verifiable | [**RLVR**](grpo-rlvr.ipynb) — no proxy, so no proxy to hack |
| You have preferences but no RL infrastructure | **DPO** — optimises the same objective in closed form. Same Goodhart risk, fewer moving parts |
| You can afford several RMs | **Reward-model ensembles**, penalising disagreement |
| The policy has drifted a long way | **Iterated RLHF** — re-annotate on the *current* policy's outputs and refit |
| Detecting hacking you have not thought of | Human review of the highest-reward samples. Cheap, and it works |

**The honest position.** Reward hacking is not a solved problem and is not close to one.
Every control here manages it rather than eliminating it: KL penalties trade away real
gains, ensembles reduce but do not remove blind spots, and iterated annotation is
expensive and always lagging the policy.

The structural fix is to need the proxy less — which is exactly the appeal of
[verifiable rewards](grpo-rlvr.ipynb), where the reward is *checked* rather than
predicted. Where verification is possible it should be preferred; where it is not, assume
your reward model will be exploited, and instrument accordingly. The single most valuable
habit is cheap and rarely done: **read the highest-scoring samples**. Reward hacking is
usually obvious the moment a human looks at the output, and invisible in every metric.

## 8. Resources

- [Deep Reinforcement Learning from Human Preferences](https://arxiv.org/abs/1706.03741) — Christiano et al., 2017; the original preference-based RL formulation.
- [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760) — Gao, Schulman & Hilton. Example 2's inverted-U measured at scale, with a functional form for the peak.
- [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) — InstructGPT; the reward-model + PPO + KL recipe in full.
- [A General Language Assistant as a Laboratory for Alignment](https://arxiv.org/abs/2112.00861) — preference modelling methodology, including annotator agreement.
- [The Effects of Reward Misspecification: Mapping and Mitigating Misaligned Models](https://arxiv.org/abs/2201.03544) — when proxy optimisation flips from helpful to harmful.
- [Reward Model Ensembles Help Mitigate Overoptimization](https://arxiv.org/abs/2310.02743) — the ensemble/uncertainty control, with ablations.
- [Length-Controlled AlpacaEval](https://arxiv.org/abs/2404.04475) — a concrete statistical correction for the length bias of Example 4.
- [Reward Hacking in Reinforcement Learning](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/) — a broad, current survey.